In [1]:
import numpy as np

import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle


In [2]:
from nmf_utils import plot_components

In [3]:
def main(path_save,file_stem,pup,to_exclude):
    """
    Apply manual exclusion of selected final components and update saved results.

    This function:
    1. Loads previously processed final component masks from a pickle bundle.
    2. Checks if the current experiment and file require component exclusion.
    3. Removes specified components based on provided indices.
    4. Updates and saves a visualization of the corrected components.
    5. Stores the corrected components back into the original bundle.

    Parameters
    ----------
    path_save : str
        Directory where the input/output pickle file is stored.
    file_stem : str
        Base filename used to locate the component bundle and save outputs.
    pup : str
        A combination of experiment and animal IDs
    to_exclude : dict
        Dictionary specifying components to remove:
        {pup: [indices_to_remove]}

    Returns
    -------
    None
        The updated bundle is saved in-place at:
        '{file_stem}_final_components.pkl'

    Outputs (updated in bundle)
    ---------------------------
    final_components_corrected : np.ndarray
        Updated component masks after removing components identified for exclusion.

    Notes
    -----
    - If no matching exclusions are found, the original components are preserved.
    - Component indices must correspond to the order in 'final_components'.
    - A visualization of corrected components is saved only when exclusions occur.
    - This step enables the second level of manual quality control of automatically extracted components.
    """    

    file_data =os.path.join(path_save, file_stem +"_final_components.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    final_components=bundle["final_components"]
    final_components_new = final_components

    if pup in to_exclude:
        indices_to_remove = to_exclude[pup]
        final_components_new = np.delete(final_components, indices_to_remove, axis=0)
        #Plot final components and save the figure
        fig=plot_components(final_components_new)
        plt.savefig(os.path.join(path_save, file_stem +"_final_components.png"), dpi=300)
        plt.close()

    # Add contours to the bundle and resave
    bundle["final_components_corrected"] = final_components_new
    with open(file_data, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

    return



In [4]:
data_source = {
"exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO":["pup2_1_spont"]
}

In [5]:
path_dist = "Data"
temp = pd.read_excel(os.path.join(path_dist, "NMF_components_to_exclude.xlsx"))
to_exclude = {
    col: temp[col].dropna().astype(int).tolist()
    for col in temp.columns
    if temp[col].dropna().shape[0] > 0
}
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        main(path_save,file_stem, pup, to_exclude)
        print(f"Processing {file_stem} from {folder_exp}")


Processing pup2_1_spont from exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO
